# Seminar 1 - A Song of Graphs and Search

---

**Course**: Graphs and Network Analysis

**Degree**: Artificial Intelligence Degree (UAB)

**Topic**: Practical seminar that includes exercises from units 1 to 6

**Activity description**: Most of us are familiar with the Game of Thrones books or series. For those who do not know it, it is a fictional series from the HBO chain, inspired by the series of novels "A Song of Ice and Fire", which tells the experiences of a group of characters from different noble houses on the fictional continent of *Westeros* to have control of the Iron Throne and rule the seven kingdoms that make up the territory. The series' success has spawned many blogs and other sources about the series, with additional resources. The graphs that we propose to use in this exercise represent the characters of the series (or books) as nodes, and their co-appearance in a scene (the weights of the edges are higher if two characters appear simultaneously more times). So we have a social network of characters. We will use these graphs to work on some of the concepts seen in the first units of the course (graph and node metrics, search and routes). Finally, we will generate synthetic graphs that simulate a realistic network.

## Qualification

**Submission**: An '.ipynb' file from the colab corresponding to each group will be delivered (this very same file, adding the code blocks and explanations that correspond to each activity). To get the file you will need to go to File --> Download. Remember that you will have to answer and analyze the different problems. Coding alone will NOT be evaluated: explaining and reasoning about the solution of the problem is essential. **You should provide explanations of the obtained results for *at least* the exercises marked with the 💬 symbol**.
The outcome of this seminar will thus be an analysis of the network at different levels: global metrics, node importance, shortest paths, random graphs, and visualization.

**Delivery form**: The work must be done in **groups of two people** and delivered through the virtual campus (in the section corresponding to Seminar 1). **Only one group member needs to submit**.

**Doubts**: For any questions, apart from class sessions, you can contact ivan.erill@uab.cat.

**Deadline**: March 10th 23:59 CET (all day). **No late submissions accepted**.

**Grade**: The average grade for all seminars will have a weight of 10% on the final grade of the course.

# Authors

**Lab group:** GrupLab-XX

**Student 1 - Name (NIU):** Anshul Satish Pandey (1801968)

**Student 2 - Name (NIU):**

## 1. Environment setup
----

The main libraries that will be used in this seminar are the following:

* [NetworkX](https://networkx.github.io/)
* [Pandas](https://pandas.pydata.org/)
* [Matplotlib](https://matplotlib.org/)
* [NumPy](https://numpy.org/)



In [ ]:
!pip install --upgrade scipy networkx

In [ ]:
!apt install libgraphviz-dev
!pip install pygraphviz

In [ ]:
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from collections import Counter

## 2. Data collection

---

This seminar is based on data from *Game of Thrones* and "A Song of Ice and Fire" curated by Andrew Beveridge. Data is available from two different github repositories:

* [Book to Network](https://github.com/mathbeveridge/asoiaf)
* [Script to Network](https://github.com/mathbeveridge/gameofthrones)

In each of them, there is a *data* folder with several *.csv* files that encode nodes and edges of different networks.

To download the data in the *colab* environment you can run the following command:

```
$ !wget https://raw.githubusercontent.com/mathbeveridge/repo_name/master/data/file_id-nodes.csv
$ !wget https://raw.githubusercontent.com/mathbeveridge/repo_name/master/data/file_id-edges.csv
```


where,

* **repo_name** is the name of the repository, *asoiaf* for the Books and *gameofthrones* for the Script.
* **file_id** is the ID of the file you can find with the link. This indicates the book or season number.

For example, to download the graph of the first season of the series, we would run:

```
$ !wget https://raw.githubusercontent.com/mathbeveridge/gameofthrones/master/data/got-s1-nodes.csv
$ !wget https://raw.githubusercontent.com/mathbeveridge/gameofthrones/master/data/got-s1-edges.csv
```

The downloaded files can be found in */content/file_name*.

For this activity, we will work with the graph generated from **all the _books_**


*  **Download only the two .csv files corresponding to the graph generated from all the books (asoiaf-all)**.

In [ ]:
!wget https://raw.githubusercontent.com/mathbeveridge/asoiaf/refs/heads/master/data/asoiaf-all-nodes.csv
!wget https://raw.githubusercontent.com/mathbeveridge/asoiaf/refs/heads/master/data/asoiaf-all-edges.csv



## 3. Data load

---

The function *csv_to_graph()* creates a NetworkX graph from the *.csv* files encoding edges and nodes.

In [ ]:
def csv_to_graph(file_id_nodes: str, file_id_edges: str, origin: str = 'book') \
                    -> nx.graph:
    """Return a nx.graph

    Build a graph given a csv file for nodes and edge.
    origin controls the source of the graph to adapt the node features.
    """

    if origin == 'book':
        key1, key2 = 'weight', 'book'
    elif origin == 'script':
        key1, key2 = 'Weight', 'Season'
    else:
        raise NameError('Unknown origin {}'.format(origin))

    nodes = pd.read_csv(file_id_nodes)
    edges = pd.read_csv(file_id_edges)

    if key2 not in edges:
        key2 = 'id'

    g = nx.Graph()
    for row in nodes.iterrows():
        g.add_node(row[1]['Id'], name=row[1]['Label'])

    for row in edges.iterrows():
        g.add_edge(row[1]['Source'],row[1]['Target'],
                   weight=1/row[1][key1], id=row[1][key2])

    return g


* **Create a NetworkX graph from the downloaded files using the `csv_to_graph` function.** [Optionally, you can repeat the process with the graph generated from the series]

In [ ]:
g_book = csv_to_graph('asoiaf-all-nodes.csv', 'asoiaf-all-edges.csv')

* **Generate a first exploratory visualization of the graph.**

In [ ]:
plt.rcParams['figure.figsize'] = [12, 12]
plt.rcParams['figure.dpi'] = 100

pos = nx.spring_layout(g_book, seed=42, k=0.3)
nx.draw(g_book,pos = pos, with_labels=False, node_color='blue', edge_color='gray', node_size=40, font_size=10)
plt.show()



## 4. General graph metrics
---

Perform a general summary of the Network properties.

* **💬  Obtain the order, size and density of the graph, as well as the average degree of its nodes.**


In [ ]:
order = g_book.order()
print(f"Order is: {order}")
size = g_book.size()
print(f"Size is: {size}")
density = nx.density(g_book)
print(f"Density is: {density}")
av_degree = 2 * size/order
print(f"Average Degree is: {av_degree}")

💬 : *The density is very low (around 0.09%), indicating that the graph is sparse.*

*The average degree of around 7 means that, on average, each node is connected with about 7 other nodes.*

* **Check that it is a connected undirected graph.**

In [ ]:
if not g_book.is_directed() and nx.is_connected(g_book):
  print('The graph is undirected and connected')

* **💬 Make a small report on the metrics of the given graph (diameter, radius, average network distance, clustering coefficient).**

In [ ]:
diameter = nx.diameter(g_book)
print (f"Diameter is: {diameter}")
radius = nx.radius(g_book)
print (f"Radius is: {radius}")
avg_dist = nx.average_shortest_path_length(g_book)
print (f"Average Network Distance is: {avg_dist}")
clus_coef = nx.average_clustering(g_book)
print (f"Clustering Coefficient is: {clus_coef}")

💬 : *A diameter of 9 means that two nodes, no matter how far they are, are connected atmost 9 connections apart.*

*A radius of 5 means that, there exists atleast one node such that it can reach all other nodes in maximum 5 steps.*

*On average any two nodes are approx. 3.4 steps apart.*

There is a 48.58% probability that two neighbours of a nodes are also neighbours.
*italicized text*


## 5. Centrality metrics: Characters' importance
---


In this section, we will study the importance of the characters according to their centrality in the graph.

* **Compute the 10 most central nodes in the network taking into account the different types of centrality (degree, betweenness, closeness and eigenvector centrality). Use also PageRank to assess importance of the characters.**

  * *centrality_bar_plot()*: Given the corresponding centrality draw a bar graph.
  * 💬 Try to reason about the changes that you observe with the different types of centrality.

In [ ]:
#centrality is a dictionary generated by networkx centrality functions
#with keys=node_ids, values=centrality_values
def centrality_bar_plot(centrality, name='betweenness', n=10):
    sorted_centrality = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:n]
    values = [item[1] for item in sorted_centrality]
    label = [item[0] for item in sorted_centrality]

    df = pd.DataFrame({'Name': label, name: values})
    ax = df.plot.bar(x='Name', y=name, rot=90)
    ax.set_title(f"Top {n} Nodes by {name.capitalize()} Centrality")
    plt.ylabel("Centrality")

In [ ]:
plt.rcParams['figure.figsize'] = [10, 4]

degree_centrality = nx.degree_centrality(g_book) # Degree Centrality
betweenness_centrality = nx.betweenness_centrality(g_book) # Betweenness Centrality
closeness_centrality = nx.closeness_centrality(g_book) # Closeness Centrality
eigen_centrality = nx.eigenvector_centrality(g_book, max_iter=1000) # Eigenvalue Centrality


centrality_bar_plot(degree_centrality, name='degree')
centrality_bar_plot(betweenness_centrality, name='betweenness')
centrality_bar_plot(closeness_centrality, name='closeness')
centrality_bar_plot(eigen_centrality, name='eigen')

plt.rcParams['figure.figsize'] = [12, 12]

In [ ]:
pagerank_centrality = nx.pagerank(g_book, alpha=0.85)
centrality_bar_plot(pagerank_centrality, name='pagerank')

💬 : *Degree centrality ranks characters by the number of distinct co-appearances. Characters like Tyrion Lannister, Jon Snow, Sansa Stark, Jaime Lannister, and Daenerys Targaryen top this list as they interact with the widest range of characters across all five books.*

*Betweenness centrality identifies characters who serve as bridges between different communities in the network. Characters with high betweenness, act as connectors, connecting otherwise distant parts of the graph.*

*Closeness centrality measures how quickly a character can reach all others via shortest paths. The top character by closeness centrality is Tyrion Lannister (approx. 0.4763), confirming that he is the most globally accessible character in the network.*

*Eigenvector centrality weights connections by the importance of neighbours. This favors characters who are connected to other highly connected characters. Major Stark and Lannister family members dominate here because they are densely interconnected with other important characters.*

*PageRank (with α=0.85) gives a ranking similar to eigenvector centrality, emphasising characters whose importance is amplified by their connections to other important characters. It provides a measure of "prestige" in the network.*

* **What is the subgraph generated by the best connected characters?**
  * Use closeness centrality to generate the graph of the 25 most central characters.

In [ ]:
#centrality is a dictionary generated by networkx centrality functions
#with keys=node_ids, values=centrality_values

def centrality_subgraph(g, centrality, name='closeness', n=25):
    closeness = nx.closeness_centrality(g)
    top_25_nodes = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:25]
    top_25_ids = [node for node, value in top_25_nodes]
    G_sub = g.subgraph(top_25_ids)
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G_sub, k=0.5, iterations=50)

    closeness_centrality = nx.closeness_centrality(g)

    node_sizes = [closeness[node] * 5000 for node in G_sub.nodes()]

    nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes, node_color='plum', alpha=0.9)
    nx.draw_networkx_edges(G_sub, pos, width=1.5, alpha=0.3, edge_color='gray')
    nx.draw_networkx_labels(G_sub, pos, font_size=10, font_weight='bold')

    plt.title("Subgraph of the 25 Most Central Characters (Closeness)")
    plt.axis('off')
    plt.show()

In [ ]:
g_subgraph = centrality_subgraph(g_book, closeness_centrality, name = "closeness", n=25 )

* **Draw this subgraph where the nodes are of size proportional to their centrality. Highlight the most central and the least central node in the graph (for instance, use the color of the node to highlight it).**
  * Use *closeness centrality* and scale it appropriately to emphasize the importance of different nodes.

In [ ]:
def draw_highlighted_subgraph(g, centrality, n=25):

    top_n_data = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:n]
    top_n_ids = [node for node, val in top_n_data]

    G_sub = g.subgraph(top_n_ids)

    sub_closeness = {node: closeness_centrality[node] for node in G_sub.nodes()}

    min_val = min(sub_closeness.values())
    max_val = max(sub_closeness.values())
    node_sizes = [300 + 2000 * (sub_closeness[n] - min_val) / (max_val - min_val)
              for n in G_sub.nodes()]

    most_central_node = top_n_data[0][0]
    least_central_node = top_n_data[-1][0]

    node_colors = []
    for node in G_sub.nodes():
        if node == most_central_node:
            node_colors.append('red')
        elif node == least_central_node:
            node_colors.append('blue')
        else:
            node_colors.append('skyblue')

    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G_sub, k=0.5, iterations=50)

    nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes,node_color=node_colors, alpha=0.9)
    nx.draw_networkx_edges(G_sub, pos, width=1.5, alpha=0.3, edge_color='gray')
    nx.draw_networkx_labels(G_sub, pos, font_size=10, font_weight='bold')

    plt.title("Subgraph of the 25 Most Central Characters (Closeness)")
    plt.axis('off')
    plt.show()

closeness_centrality = nx.closeness_centrality(g_book)
draw_highlighted_subgraph(g_book, closeness_centrality, n=25)

* **Draw the tree that the BFS and DFS algorithm would generate to traverse the graph starting from the least central node of the network according to *closeness centrality*.**
  * Use *closeness centrality* and scale it appropriately to emphasize the importance of different nodes.
  * To get the positions of the nodes, you can use the `graphviz_layout(tree, prog='dot')` command.
  * 💬 Comment on the obtained result.


In [ ]:
closeness = nx.closeness_centrality(g_book)
start_node = min(closeness, key=closeness.get)

bfs_tree = nx.bfs_tree(g_book, source=start_node)
dfs_tree = nx.dfs_tree(g_book, source=start_node)

def draw_traversal_tree(tree, title, centrality_dict):
    plt.figure(figsize=(25, 15))
    pos = graphviz_layout(tree, prog='dot', args='-Granksep=3 -Gnodesep=0.5')

    node_sizes = [(centrality_dict[node]**2) * 15000 for node in tree.nodes()]

    node_colors = ['red' if node == start_node else 'skyblue' for node in tree.nodes()]

    nx.draw(tree, pos, node_size=node_sizes, node_color=node_colors, with_labels=False, width=0.8, edge_color = "gray", arrows=True, alpha=0.8, edgecolors='black', linewidths=1.5)

    plt.title(title)
    plt.show()

draw_traversal_tree(bfs_tree, f"BFS Tree starting from {start_node}", closeness)


In [ ]:
draw_traversal_tree(dfs_tree, f"DFS Tree starting from {start_node}", closeness)

💬 : *The BFS tree looks very wide as it explores all immediate neighbors first before moving deeper. It expands level by level, first visiting all of Gormon Tyrell's direct neighbours, then all of their unexplored neighbours, and so on.*

*The DFS tree is narrow and deep. Starting from Gormon Tyrell, the DFS tree plunges as far as possible along one path before backtracking, producing long chains of nodes that can iterate almost the entire network in a single branch.*

*The key structural difference is that BFS preserves distance information, while DFS does not.*

* **💬 Compute the shortest path between the least and the most central nodes in the complete graph.**

In [ ]:
most_central = max(closeness_centrality, key=closeness_centrality.get)
least_central = min(closeness_centrality, key=closeness_centrality.get)

print(f"Least Central: {least_central}")
print(f"Most Central: {most_central}")

path = nx.shortest_path(g_book, source=least_central, target=most_central)
distance = nx.shortest_path_length(g_book, source=least_central, target=most_central)

print(f"Shortest Path: {' --> '.join(path)}")
print(f"Path Length: {distance}")

💬 : *Here, even the least connected node is 6 steps away from the most central node. It suggests that the graph is well connected and there does not exists a very isolated community of nodes.*

## 6. Random graph models
----
Up to this point, we have worked with a graph generated from the data extracted from the *Song of Ice and Fire* books. In the real world, however, obtaining the data needed to construct this graph can become very complex and expensive. This is one of the reasons why, over time, the synthetic generation of graphs has been studied.

In this section we will work on the different models described in class. We will generate random graphs and study their properties.

* **Generate random graphs with the Uniform, Gilbert and Barabási-Albert models. Fix the number of nodes to the order of the GoT graph. Adjust the rest of the parameters of the graph generation function to obtain graphs with similar number of edges.**

### Erdös-Rény: Uniform Model (gnm)

In [ ]:
n = g_book.number_of_nodes()
e = g_book.number_of_edges()

g_uniform = nx.gnm_random_graph(n, e)

### Erdös-Rény: Gilbert Model (gnp)


In [ ]:
p = (2 * e) / (n * (n - 1))
g_gilbert = nx.gnp_random_graph(n, p)

### Barabási-Albert Model



In [ ]:
k = round(e / n)
g_barbasi = nx.barabasi_albert_graph(n, k)

In [ ]:
g_dict = {'Book': g_book, 'Uniform': g_uniform, 'Erdos': g_gilbert, 'Barbasi': g_barbasi}

* **💬 Show the order and size of the graph as well as the average degree and clustering coefficient of its nodes. Compute also the intervals between the maximum and minimum centralities for each family of synthetic graphs. Make a small report of the main metrics. Which random graph resembles more closely the graph from the books?**
     * You can set the graph generation using a random seed. This way, two different runs will generate exactly the same graph.

In [ ]:
for k, g in g_dict.items():
    nodes = g.number_of_nodes()
    edges = g.number_of_edges()
    avg_deg = (2 * edges) / nodes
    avg_clust = nx.average_clustering(g)

    cent = nx.degree_centrality(g)
    c_vals = list(cent.values())
    c_interval = max(c_vals) - min(c_vals)

    print(f"Graph: {k:<8}, Order: {nodes:<5}, Size: {edges:<6}, Avg Degree: {avg_deg:<7}, Clustering: {avg_clust:<20}, Cent. Interval: {c_interval}")

💬 : *The Barabasi-Albert model is the comparatively the most similar to the book network, but it is still an imperfect match. It's able to capture the high centrality interval compared to any other model. However, it cannot replicate the clustering. Almost all random graph models struggle with this. The Barabasi model overshoots the edge count as well.*

* **💬 Check whether the networks (the three randomly generated ones and the network extracted from the books) follow a Power Law.**

In [ ]:
plt.rcParams['figure.figsize'] = [13, 5]

fig, axes = plt.subplots(1, len(g_dict), figsize=(16, 4))

for idx, (k, g) in enumerate(g_dict.items()):
    degrees = [d for _, d in g.degree()]
    degree_count = Counter(degrees)

    deg_values = sorted(degree_count.keys())
    deg_freq = [degree_count[d] / len(degrees) for d in deg_values]

    axes[idx].loglog(deg_values, deg_freq, 'bo', markersize=4)
    axes[idx].set_xlabel('Degree (log)')
    axes[idx].set_ylabel('P(k) (log)')
    axes[idx].set_title(f'{k}')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Degree Distribution — Log-Log Scale (Power Law Check)')
plt.tight_layout()
plt.show()

💬 : *If a networks follow the power law, then they would appear as a straight line on the plot. The uniform and gilbert networks fail this, and they do not follow the power law. The Barabasi - Albert model follows this law as visible in the plot.*